# GRPO post-training for Python code generation

This notebook reproduces the `code-grpo-humaneval` experiment on a hosted Colab GPU. It measures a frozen baseline, runs a one-step GRPO smoke test, optionally trains a LoRA adapter, and evaluates the adapter on a held-out HumanEval split.

**Security:** generated Python runs inside the disposable Colab VM. Do not mount Google Drive and do not add API keys while evaluation or training is running.

## 1. Confirm the hosted GPU
Choose **Runtime > Change runtime type > GPU** before running this cell. A T4 (16 GB) or better is recommended.

In [1]:
!nvidia-smi
import torch

assert torch.cuda.is_available(), 'No GPU detected. Select a GPU runtime and reconnect.'
props = torch.cuda.get_device_properties(0)
print(f'GPU: {props.name} | VRAM: {props.total_memory / 2**30:.1f} GB')

Sat Aug  1 09:34:43 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   43C    P8              9W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

## 2. Clone the public repository

In [2]:
import os
import subprocess
from pathlib import Path

repo = Path('/content/code-grpo-humaneval')
if repo.exists():
    subprocess.run(['git', '-C', str(repo), 'pull', '--ff-only'], check=True)
else:
    clone_url = 'https://github.com/Hamza-Nadif/code-grpo-humaneval.git'
    subprocess.run(['git', 'clone', clone_url, str(repo)], check=True)
os.chdir(repo)
print('Commit:', subprocess.check_output(['git', 'rev-parse', 'HEAD'], text=True).strip())

Commit: 240e4fd17d491634f4c1559c2cf235e4951c2eb0


## 3. Install and verify dependencies
This installation is stored only in the temporary Colab runtime.

In [3]:
%pip install -q -r requirements.txt -r requirements-dev.txt

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 889.0/889.0 kB 34.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 529.0/529.0 kB 47.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.9/40.9 MB 21.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.5/11.5 MB 98.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.1/50.1 MB 18.2 MB/s eta 0:00:00


In [4]:
import importlib.metadata as metadata

for package in ['torch', 'transformers', 'datasets', 'trl', 'peft', 'bitsandbytes']:
    print(f'{package}: {metadata.version(package)}')
!pytest
!ruff check .

torch: 2.11.0+cu128
transformers: 5.13.1
datasets: 4.8.5
trl: 1.9.2
peft: 0.19.1
bitsandbytes: 0.50.0
.................                                                        [100%]
17 passed in 0.57s
All checks passed!


## 4. Build deterministic HumanEval splits
The script downloads only the small dataset from a pinned OpenAI commit and writes 120 training, 22 validation, and 22 held-out test tasks.

In [5]:
!python build_training_data.py --output-dir data
!cat data/manifest.json | head -n 20

{
  "manifest": "data/manifest.json",
  "sizes": {
    "train": 120,
    "validation": 22,
    "test": 22
  }
}
{
  "dataset_id": "openai/human-eval",
  "source_url": "https://raw.githubusercontent.com/openai/human-eval/6d43fb980f9fee3c892a914eda09951f772ad10d/data/HumanEval.jsonl.gz",
  "revision": "6d43fb980f9fee3c892a914eda09951f772ad10d",
  "seed": 42,
  "total_rows": 164,
  "warning": "These are internal HumanEval splits. Results are not comparable to the official 164-task HumanEval benchmark after training on the train split.",
  "files": {
    "official_full": {
      "path": "data/humaneval_official_full.jsonl",
      "rows": 164,
      "sha256": "86bbccf3c59e2db777d3f698892402fb05bed89c6de83d23bda12ada2bbd747a",
      "task_ids": [
        "HumanEval/0",
        "HumanEval/1",
        "HumanEval/2",
        "HumanEval/3",
        "HumanEval/4",
        "HumanEval/5",
        "HumanEval/6",


## 5. Validate the execution harness
Canonical solutions should obtain pass@1 = 1.0. This is a pipeline check, not a model result.

In [ ]:
!python evaluate_baseline.py \
  --data data/humaneval_test.jsonl \
  --backend oracle \
  --executor local \
  --allow-local-code-execution \
  --output-dir results/oracle-smoke

## 6. Measure the frozen baseline
This downloads `Qwen2.5-Coder-0.5B-Instruct` and evaluates one deterministic completion on each of the 22 held-out tasks. The result is the before-GRPO score.

In [ ]:
!python evaluate_baseline.py \
  --data data/humaneval_test.jsonl \
  --backend transformers \
  --model Qwen/Qwen2.5-Coder-0.5B-Instruct \
  --quantization 4bit \
  --samples-per-task 1 \
  --temperature 0 \
  --executor local \
  --allow-local-code-execution \
  --output-dir results/baseline-heldout

In [10]:
%cd /content/code-grpo-humaneval
!git pull origin main
!git rev-parse --short HEAD


/content
remote: Enumerating objects: 14, done.
remote: Counting objects: 100% (14/14), done.
remote: Compressing objects: 100% (3/3), done.
remote: Total 8 (delta 5), reused 8 (delta 5), pack-reused 0 (from 0)
Unpacking objects: 100% (8/8), 1.61 KiB | 550.00 KiB/s, done.
From https://github.com/Hamza-Nadif/code-grpo-humaneval
 * branch            main       -> FETCH_HEAD
   230af19..588f141  main       -> origin/main
Updating 230af19..588f141
Fast-forward
 README.md                                 |  1 +
 notebooks/code_grpo_humaneval_colab.ipynb |  2 ++
 tests/test_train_grpo.py                  | 25 ++++++++++++++++
 train_grpo.py                             | 47 +++++++++++++++++++++++++++++--
 4 files changed, 72 insertions(+), 3 deletions(-)
 create mode 100644 tests/test_train_grpo.py
588f141


In [11]:
!python train_grpo.py \
  --model Qwen/Qwen2.5-Coder-0.5B-Instruct \
  --train-data data/humaneval_train.jsonl \
  --eval-data data/humaneval_validation.jsonl \
  --quantization 4bit \
  --precision fp16 \
  --num-generations 2 \
  --gradient-accumulation-steps 2 \
  --max-completion-length 128 \
  --max-steps 1 \
  --executor local \
  --allow-local-code-execution \
  --output-dir outputs/qwen-code-grpo-smoke

{
  "train": {
    "rows": 120,
    "columns": [
      "canonical_solution",
      "entry_point",
      "prompt",
      "starter_code",
      "task_id",
      "test"
    ],
    "path": "data/humaneval_train.jsonl"
  },
  "validation": {
    "rows": 22,
    "columns": [
      "canonical_solution",
      "entry_point",
      "prompt",
      "starter_code",
      "task_id",
      "test"
    ],
    "path": "data/humaneval_validation.jsonl"
  },
  "configuration": {
    "train_data": "data/humaneval_train.jsonl",
    "eval_data": "data/humaneval_validation.jsonl",
    "model": "Qwen/Qwen2.5-Coder-0.5B-Instruct",
    "output_dir": "outputs/qwen-code-grpo-smoke",
    "max_steps": 1,
    "learning_rate": 5e-06,
    "num_generations": 2,
    "gradient_accumulation_steps": 2,
    "max_completion_length": 128,
    "temperature": 0.8,
    "timeout": 3.0,
    "executor": "local",
    "lora_r": 16,
    "lora_alpha": 32,
    "quantization": "4bit",
    "precision": "fp16",
    "seed": 42,
    "dry_ru

## 7. One-step GRPO smoke test
This confirms that model loading, 4-bit QLoRA, generation, rewards, backpropagation, and adapter saving work together. It is not the final experiment.

In [9]:
!python train_grpo.py \
  --model Qwen/Qwen2.5-Coder-0.5B-Instruct \
  --train-data data/humaneval_train.jsonl \
  --eval-data data/humaneval_validation.jsonl \
  --quantization 4bit \
  --num-generations 2 \
  --gradient-accumulation-steps 2 \
  --max-completion-length 128 \
  --max-steps 1 \
  --executor local \
  --allow-local-code-execution \
  --output-dir outputs/qwen-code-grpo-smoke

{
  "train": {
    "rows": 120,
    "columns": [
      "canonical_solution",
      "entry_point",
      "prompt",
      "starter_code",
      "task_id",
      "test"
    ],
    "path": "data/humaneval_train.jsonl"
  },
  "validation": {
    "rows": 22,
    "columns": [
      "canonical_solution",
      "entry_point",
      "prompt",
      "starter_code",
      "task_id",
      "test"
    ],
    "path": "data/humaneval_validation.jsonl"
  },
  "configuration": {
    "train_data": "data/humaneval_train.jsonl",
    "eval_data": "data/humaneval_validation.jsonl",
    "model": "Qwen/Qwen2.5-Coder-0.5B-Instruct",
    "output_dir": "outputs/qwen-code-grpo-smoke",
    "max_steps": 1,
    "learning_rate": 5e-06,
    "num_generations": 2,
    "gradient_accumulation_steps": 2,
    "max_completion_length": 128,
    "temperature": 0.8,
    "timeout": 3.0,
    "executor": "local",
    "lora_r": 16,
    "lora_alpha": 32,
    "quantization": "4bit",
    "seed": 42,
    "dry_run": false,
    "allow_loc

## 8. Main training run
Keep the switch at `False` until the smoke test succeeds. Start with 10 steps; increase to 50 only after checking runtime and GPU memory.

In [7]:
RUN_MAIN_TRAINING = False
MAIN_STEPS = 10

if RUN_MAIN_TRAINING:
    command = [
        'python', 'train_grpo.py',
        '--model', 'Qwen/Qwen2.5-Coder-0.5B-Instruct',
        '--train-data', 'data/humaneval_train.jsonl',
        '--eval-data', 'data/humaneval_validation.jsonl',
        '--quantization', '4bit',
        '--num-generations', '4',
        '--gradient-accumulation-steps', '4',
        '--max-completion-length', '256',
        '--max-steps', str(MAIN_STEPS),
        '--executor', 'local',
        '--allow-local-code-execution',
        '--output-dir', 'outputs/qwen-code-grpo',
    ]
    subprocess.run(command, check=True)
else:
    print('Main training is disabled. Set RUN_MAIN_TRAINING = True after the smoke test succeeds.')

Main training is disabled. Set RUN_MAIN_TRAINING = True after the smoke test succeeds.


## 9. Evaluate the trained adapter
This cell uses the main adapter when available; otherwise it evaluates the one-step smoke adapter only to verify the loading path.

In [ ]:
adapter = Path('outputs/qwen-code-grpo')
if not adapter.exists():
    adapter = Path('outputs/qwen-code-grpo-smoke')
print('Evaluating adapter:', adapter)
command = [
    'python', 'evaluate_baseline.py',
    '--data', 'data/humaneval_test.jsonl',
    '--backend', 'transformers',
    '--model', 'Qwen/Qwen2.5-Coder-0.5B-Instruct',
    '--adapter', str(adapter),
    '--quantization', '4bit',
    '--samples-per-task', '1',
    '--temperature', '0',
    '--executor', 'local',
    '--allow-local-code-execution',
    '--output-dir', 'results/grpo-heldout',
]
subprocess.run(command, check=True)

## 10. Compare and download results

In [ ]:
import json


def load_summary(path):
    return json.loads(Path(path).read_text())

baseline = load_summary('results/baseline-heldout/summary.json')
trained = load_summary('results/grpo-heldout/summary.json')
before = baseline['metrics']['pass@1']
after = trained['metrics']['pass@1']
print(f'Baseline pass@1: {before:.4f}')
print(f'GRPO pass@1:     {after:.4f}')
print(f'Difference:      {after - before:+.4f}')

In [ ]:
import shutil

from google.colab import files

bundle = Path('/content/code-grpo-experiment')
if bundle.exists():
    shutil.rmtree(bundle)
bundle.mkdir()
shutil.copytree('results', bundle / 'results')
shutil.copytree(adapter, bundle / 'adapter')
shutil.copy('data/manifest.json', bundle / 'data_manifest.json')
archive = shutil.make_archive('/content/code-grpo-experiment', 'zip', bundle)
print('Archive:', archive)
files.download(archive)